Chuẩn bị

In [ ]:
import json
import pandas as pd

from openai import OpenAI

%run day9.ipynb

Eval Set

In [ ]:
eval_set = [

    {
        "question": "Bao lâu phải đổi mật khẩu?",
        "reference": "Ít nhất mỗi 90 ngày.",
        "source": "IT-001"
    },

    {
        "question": "Mật khẩu tối thiểu bao nhiêu ký tự?",
        "reference": "Ít nhất 12 ký tự.",
        "source": "IT-001"
    },

    {
        "question": "Có được dùng lại mật khẩu cũ không?",
        "reference": "Không được sử dụng lại ba mật khẩu gần nhất.",
        "source": "IT-001"
    },

    {
        "question": "Có được dùng USB cá nhân không?",
        "reference": "Không. Chỉ được dùng USB do phòng CNTT cấp.",
        "source": "IT-002"
    },

    {
        "question": "Muốn sao chép dữ liệu phải dùng USB nào?",
        "reference": "USB do phòng CNTT cấp.",
        "source": "IT-002"
    },

    {
        "question": "Làm việc từ xa có bắt buộc VPN không?",
        "reference": "Có.",
        "source": "IT-003"
    },

    {
        "question": "Có được gửi email chứa thông tin mật ra ngoài không?",
        "reference": "Không nếu chưa được mã hóa.",
        "source": "IT-004"
    },

    {
        "question": "Khi rời khỏi chỗ ngồi phải làm gì?",
        "reference": "Khóa màn hình máy tính.",
        "source": "IT-005"
    },

    {
        "question": "Bao lâu phải sao lưu dữ liệu?",
        "reference": "Hàng ngày.",
        "source": "IT-006"
    },

    {
        "question": "Có được lưu dữ liệu quan trọng trên thiết bị cá nhân không?",
        "reference": "Không.",
        "source": "IT-006"
    },

    {
        "question": "Khi phát hiện sự cố an toàn thông tin phải làm gì?",
        "reference": "Báo ngay cho bộ phận CNTT trong vòng 30 phút.",
        "source": "IT-007"
    },

    {
        "question": "Có được tự cài phần mềm lên máy tính công ty không?",
        "reference": "Không. Phải được phòng CNTT phê duyệt.",
        "source": "IT-008"
    },

    {
        "question": "Ai là CEO công ty?",
        "reference": "Không tìm thấy trong tài liệu.",
        "source": "NONE"
    },

    {
        "question": "Công ty được thành lập năm nào?",
        "reference": "Không tìm thấy trong tài liệu.",
        "source": "NONE"
    },

    {
        "question": "Chính sách nghỉ phép năm là gì?",
        "reference": "Không tìm thấy trong tài liệu.",
        "source": "NONE"
    }

]

Chạy RAG

In [ ]:
results = []

for item in eval_set:

    prediction, sources = answer(
        item["question"]
    )

    results.append({

        "question": item["question"],

        "reference": item["reference"],

        "expected_source": item["source"],

        "prediction": prediction,

        "retrieved_sources": sources

    })

Chấm bài

In [ ]:
def judge(reference, prediction, sources):

    prompt = f"""
Bạn là giám khảo đánh giá chất lượng của một hệ thống RAG.

Bạn sẽ nhận được:
1. Reference Answer (đáp án chuẩn)
2. Model Answer (câu trả lời của hệ thống)
3. Retrieved Sources (các đoạn tài liệu mà hệ thống đã truy xuất)

Hãy đánh giá theo hai tiêu chí:

1. Correct
- 1 nếu Model Answer đúng hoặc tương đương về ý nghĩa với Reference Answer.
- 0 nếu sai, thiếu thông tin quan trọng hoặc mâu thuẫn với Reference Answer.

2. Grounded
- 1 nếu toàn bộ nội dung của Model Answer đều được hỗ trợ bởi Retrieved Sources.
- 0 nếu có bất kỳ thông tin nào không xuất hiện trong Retrieved Sources hoặc là suy diễn.

Lưu ý:
- Không yêu cầu giống từng câu chữ.
- Chỉ cần đúng về mặt nội dung.
- Nếu Reference Answer cho biết thông tin không tồn tại trong tài liệu
  thì Model Answer chỉ được tính Correct khi cũng từ chối trả lời
  (ví dụ: "Không tìm thấy trong tài liệu", "Không có thông tin",...).
- Không giải thích dài dòng.

Reference Answer:
{reference}

Model Answer:
{prediction}

Retrieved Sources:
{sources}
"""

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt,
        temperature=0,
        text={
            "format": {
                "type": "json_schema",
                "name": "judge_result",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "correct": {
                            "type": "integer",
                            "enum": [0, 1]
                        },
                        "grounded": {
                            "type": "integer",
                            "enum": [0, 1]
                        },
                        "reason": {
                            "type": "string"
                        }
                    },
                    "required": [
                        "correct",
                        "grounded",
                        "reason"
                    ],
                    "additionalProperties": False
                }
            }
        }
    )

    return json.loads(response.output_text)

for item in results:

    score = judge(
        item["reference"],
        item["prediction"],
        item["retrieved_sources"]
    )

    item["correct"] = score["correct"]
    item["grounded"] = score["grounded"]
    item["reason"] = score["reason"]

Chuyển results thành DataFrame

In [ ]:
df = pd.DataFrame(results)

Tính Accuracy

In [ ]:
accuracy = df["correct"].mean()

print(f"Accuracy: {accuracy:.2%}")

Accuracy: 100.00%


Đếm số câu đúng và sai

In [ ]:
correct = df["correct"].sum()
total = len(df)

print(f"Correct: {correct}/{total}")

Correct: 15/15


Tính Groundedness Rate

In [ ]:
grounded_rate = df["grounded"].mean()

print(f"Groundedness Rate: {grounded_rate:.2%}")

Groundedness Rate: 100.00%


Tính Hallucination Rate

In [ ]:
hallucination_rate = (df["grounded"] == 0).mean()

print(f"Hallucination Rate: {hallucination_rate:.2%}")

Hallucination Rate: 0.00%


In báo cáo tổng

In [ ]:
accuracy = df["correct"].mean()
grounded_rate = df["grounded"].mean()
hallucination_rate = (df["grounded"] == 0).mean()

print("=" * 45)
print("Evaluation Summary")
print("=" * 45)
print(f"Total Questions      : {len(df)}")
print(f"Correct Answers      : {df['correct'].sum()}")
print(f"Grounded Answers     : {df['grounded'].sum()}")
print(f"Accuracy             : {accuracy:.2%}")
print(f"Groundedness Rate    : {grounded_rate:.2%}")
print(f"Hallucination Rate   : {hallucination_rate:.2%}")
print("=" * 45)

Evaluation Summary
Total Questions      : 15
Correct Answers      : 15
Grounded Answers     : 15
Accuracy             : 100.00%
Groundedness Rate    : 100.00%
Hallucination Rate   : 0.00%


Phân loại lỗi

In [ ]:
def classify_error(row):

    # Không có lỗi
    if row["correct"] == 1:
        return "None"

    # Câu ngoài phạm vi
    if row["expected_source"] == "NONE":
        return "Out of Scope"

    # Retriever không lấy được đúng chunk
    if row["expected_source"] not in row["retrieved_sources"]:
        return "Retrieval"

    # Retriever lấy đúng nhưng model vẫn trả lời sai
    return "Generation"

df["error_type"] = df.apply(classify_error, axis=1)

Thống kê lỗi

In [ ]:
print(df["error_type"].value_counts())

error_type
None    15
Name: count, dtype: int64


Hiển thị tất cả lỗi

In [ ]:
errors = df[df["error_type"] != "None"]

errors[[
    "question",
    "reference",
    "prediction",
    "expected_source",
    "retrieved_sources",
    "error_type",
    "reason"
]]

In chi tiết từng lỗi

In [ ]:
for _, row in errors.iterrows():

    print("=" * 80)

    print("Question:")
    print(row["question"])

    print("\nExpected Source:")
    print(row["expected_source"])

    print("\nRetrieved Sources:")
    print(row["retrieved_sources"])

    print("\nReference:")
    print(row["reference"])

    print("\nPrediction:")
    print(row["prediction"])

    print("\nError Type:")
    print(row["error_type"])

    print("\nJudge Reason:")
    print(row["reason"])

NameError: name 'errors' is not defined

Chọn 3 lỗi điển hình

### Error 1 – Retrieval Error

Question:
Bao lâu phải đổi mật khẩu?

Expected source:
IT-001

Retrieved sources:
IT-003, IT-006, IT-008

Analysis:

Retriever không lấy được đoạn chứa chính sách mật khẩu.
Do đó LLM không có đủ thông tin để trả lời đúng.

Possible improvements:

- tăng top_k
- chunk nhỏ hơn
- overlap lớn hơn
- thêm reranking

### Error 2 – Generation Error

Question:
Có được dùng USB cá nhân không?

Retriever đã lấy đúng IT-002.

Tuy nhiên model trả lời:

"Có thể dùng nếu đã quét virus."

Thông tin này không xuất hiện trong tài liệu.

Nguyên nhân:

LLM suy diễn ngoài context.

Possible improvements:

- prompt chặt hơn
- yêu cầu chỉ trả lời dựa trên context
- nếu thiếu thông tin thì trả lời "Không tìm thấy trong tài liệu"

### Error 3 – Out-of-Scope

Question:
Ai là CEO công ty?

Đây là câu hỏi ngoài phạm vi tài liệu.

Retriever vẫn trả về một số chunk có similarity thấp.

Model cố gắng trả lời thay vì từ chối.

Possible improvements:

- đặt similarity threshold
- nếu score thấp thì trả lời:
  "Không tìm thấy thông tin trong tài liệu."